# Guardrail 9 — Groundedness

**Where it sits:** between the LLM's raw output and the output guard (#7). Sits *after* the model has emitted, *before* anything reaches the user.

**What it stops:** hallucinations — claims the model made that aren't actually supported by the retrieved chunks. Distinct from the citation check in #7, which only verifies the citation ID exists; this guard verifies the *content* of the claim is in the cited chunk.

**Why it matters:** a model can cite `d1` correctly and still hallucinate. The chunk says "Paris is the capital of France" but the model claims "Paris is the capital of France and has 12 million residents." Citation passes; groundedness fails.

**Decision contract:** `{allow | rewrite | block, grounded_response, dropped_claims[], reasons[]}`

**Self-contained:** inlines a toy RAG. No imports from other folders.

## Step 1 — toy RAG (with a model that hallucinates)

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the same directory as this notebook (works in Jupyter)
_env_path = Path.cwd() / ".env"
if not _env_path.exists():
    _env_path = Path(__file__).parent / ".env" if "__file__" in globals() else _env_path
load_dotenv(_env_path, override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

# LangChain primitives, all driven by .env
LLM_MODEL     = os.getenv("MINIMAX_MODEL", "MiniMax-M3")
LLM_BASE_URL  = os.getenv("MINIMAX_BASE_URL", "https://api.minimax.io/v1")
LLM_API_KEY   = os.getenv("MINIMAX_API_KEY", "")

llm = ChatOpenAI(
    model=LLM_MODEL,
    api_key=LLM_API_KEY or "sk-fake",   # placeholder if no key -- calls will fail loudly
    base_url=LLM_BASE_URL,
    temperature=0,
)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=LLM_API_KEY or "sk-fake",
    base_url=LLM_BASE_URL,
)

print(f"LLM configured:  model={LLM_MODEL}  base_url={LLM_BASE_URL}")
print(f"API key loaded:  {'yes ('+LLM_API_KEY[:8]+'...)' if LLM_API_KEY else 'NO -- calls will fail; toy fallbacks below are unaffected'}")

# A safe wrapper so toy guardrail tests below stay deterministic.
# If FAKE_LLM=1 (or no key), use the toy. Otherwise call the real one.
_USE_FAKE = os.getenv("FAKE_LLM", "1") == "1" or not LLM_API_KEY

def chat(prompt: str, system: str = None) -> str:
    """Invoke the LangChain ChatOpenAI. Returns .content."""
    if _USE_FAKE:
        raise RuntimeError("chat() called but FAKE_LLM=1 -- use the toy LLM in this notebook's tests")
    msgs = []
    if system:
        msgs.append(SystemMessage(content=system))
    msgs.append(HumanMessage(content=prompt))
    return llm.invoke(msgs).content


In [ ]:
import re

DOCS = [
    {"id": "d1", "text": "The capital of France is Paris."},
    {"id": "d2", "text": "The capital of Japan is Tokyo."},
    {"id": "d3", "text": "Paris has a population of about 2.1 million."},
    {"id": "d4", "text": "Tokyo has a population of about 14 million."},
]

def llm(query, scenario):
    """Toy LLM with deliberate hallucination patterns."""
    return {
        "fully_grounded":   "Paris is the capital of France. (source: d1)",

        "partial":          "Paris is the capital of France. It is also the largest city in Europe with 12 million residents. (source: d1)",

        "fully_hallucinated": "Paris is the capital of France. The Eiffel Tower was built in 1492 by Napoleon. (source: d1)",

        "numbers_wrong":    "The population of Tokyo is 50 million. (source: d4)",

        "citation_only":    "Paris is in France. (source: d1)",   # cited correctly but d1 doesn't say 'in France'
    }[scenario]

## Step 2 — groundedness guardrail

In [ ]:
STOPWORDS = {"the","a","an","is","are","was","were","be","been","being",
             "of","in","on","at","to","for","and","or","but","it","its",
             "this","that","these","those","with","as","by","from","has","have"}

def content_words(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS]

def overlap_ratio(claim_words, chunk_words):
    if not claim_words:
        return 0.0
    claim_set = set(claim_words)
    return len(claim_set & set(chunk_words)) / len(claim_set)

def groundedness_guard(response: str, chunks_seen: list,
                       overlap_threshold: float = 0.6,
                       require_full_grounding: bool = True):
    """Strip the (source: X) tag and split into sentences; flag the
    ungrounded ones."""
    # 1. strip the citation tag — we don't want to test overlap against it
    body = re.sub(r"\(source:\s*[^)]+\)", "", response).strip()

    # 2. split into sentences
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", body) if s.strip()]

    # 3. score each sentence against each chunk
    grounded, ungrounded = [], []
    chunk_words_cache = [(c, content_words(c["text"])) for c in chunks_seen]

    for sent in sentences:
        sw = content_words(sent)
        if not sw:
            grounded.append(sent)
            continue
        # best-overlap chunk
        best_score, best_chunk = 0.0, None
        for c, cw in chunk_words_cache:
            s = overlap_ratio(sw, cw)
            if s > best_score:
                best_score, best_chunk = s, c

        if best_score >= overlap_threshold:
            grounded.append(f"{sent} (grounded in {best_chunk['id']}, overlap={best_score:.2f})")
        else:
            ungrounded.append({"claim": sent,
                                "best_chunk": best_chunk["id"] if best_chunk else None,
                                "best_overlap": round(best_score, 2),
                                "reason": "below_overlap_threshold"})

    reasons = []
    if ungrounded:
        reasons.append(f"ungrounded_claims:{len(ungrounded)}")

    decision = "allow"
    final = " ".join(grounded)
    if ungrounded:
        if require_full_grounding and len(ungrounded) == len(sentences):
            # nothing survived → block
            decision = "block"
        else:
            # some survived → rewrite, dropping the ungrounded
            decision = "rewrite"

    return {"decision": decision,
            "grounded_response": final,
            "dropped_claims": ungrounded,
            "reasons": reasons}

## Step 3 — test cases

In [ ]:
chunks_seen = [{"id": d["id"], "text": d["text"]} for d in DOCS]

scenarios = [
    ("fully grounded",          "fully_grounded"),
    ("partial (one claim true, one false)",  "partial"),
    ("fully hallucinated",       "fully_hallucinated"),
    ("numbers wrong",            "numbers_wrong"),
    ("citation correct, claim not in chunk",  "citation_only"),
]

for label, scenario in scenarios:
    raw = llm("...", scenario)
    r = groundedness_guard(raw, chunks_seen)
    print(f"\n=== {label} ===")
    print(f"  raw     : {raw}")
    print(f"  decision: {r['decision']}")
    print(f"  grounded: {r['grounded_response']}")
    print(f"  dropped : {[(d['claim'][:60], d['best_overlap']) for d in r['dropped_claims']]}")

## Step 4 — why this is different from citation check

In [ ]:
print("""
Citation guard (#7) answers:
  'Did the model cite a chunk that survived guard #5?'

Groundedness guard (#9) answers:
  'Is the actual content of the model's claim IN the cited chunk?'

Failure modes only #9 catches:
  • numeric hallucination  — model says '50 million' when chunk says '14 million'
  • entity hallucination   — model attributes a fact to the wrong entity
  • composition           — model says 'A and B' when chunk has A but not B
  • paraphrase drift      — model paraphrases so far that the meaning changes

Industry implementations:
  • Vectara HHEM leaderboard — open hallucination-detection benchmark
  • Galileo, Patronus, AWS Ground Truth — production hallucination classifiers
  • Self-check / NLI-based — feed (premise=chunk, hypothesis=claim) to an
    NLI model; if hypothesis is not entailed by premise, drop it

The toy here uses word-overlap as a stand-in for NLI. Real systems use
either a small NLI model (DeBERTa-v3-base-NLI) or an LLM self-check.
""")

In [ ]:
### Real LangChain demo: groundedness guard as a Runnable on retrieved docs + LLM output

from langchain_core.runnables import RunnableLambda

def _grounded_chain(retriever):
    def _run(query: str):
        docs = retriever.invoke(query)
        chunks = [{"id": d.metadata.get("id", "?"), "text": d.page_content} for d in docs]
        prompt = f"Answer using ONLY these docs:\\n\\n" + "\\n".join(c["text"] for c in chunks) + f"\\n\\nQ: {query}\\nA:"
        if not _USE_FAKE:
            ans = llm.invoke(prompt).content
        else:
            ans = "Paris."
        return groundedness_guard(ans, chunks)
    return _run

print(f"groundedness Runnable constructed; pass a retriever to use it. FAKE_LLM={_USE_FAKE}")


## Takeaways

- **Citation ≠ groundedness.** A model can cite the right chunk and still hallucinate within it. You need both guards.
- **Sentence-level, not whole-response.** A response can have 3 true claims and 1 hallucinated one. Drop the 1, keep the 3.
- **NLI > word-overlap in production.** Word overlap misses paraphrases and catches unrelated words by accident. An NLI model (`premise=chunk, hypothesis=claim`) is the right tool.
- **Don't block on partial hallucination — rewrite.** The user got *something* useful. Strip the bad sentence, keep the rest, surface what was dropped.
- **Tune the threshold on a labeled set.** 0.6 is a starting point; the right value depends on your chunk sizes and the LLM you use.

**Negative fixture checklist:** numeric hallucination, entity swap, composed claim (A+B from A only), paraphrase drift. ✓